# 🏢 AI Company Brochure Generator
This notebook allows you to generate a professional company brochure with customized formatting (bullet points vs paragraphs), tone selection, and export options for PDF and DOCX.

In [ ]:
# Install required libraries if not already present
!pip3 -q install gradio openai beautifulsoup4 requests python-docx fpdf2 python-dotenv

In [ ]:
import os
import json
import logging
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin, urlparse
from fpdf import FPDF
from docx import Document
from docx.shared import Pt

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 1. INITIALIZATION
load_dotenv(override=True)

BASE_URL = os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1")
api_key = os.getenv("GROQ_API_KEY")
MODEL = os.getenv("GROQ_MODEL", 'openai/gpt-oss-120b')

if not api_key:
    print("⚠️ WARNING: GROQ_API_KEY not found in environment variables.")

client = OpenAI(base_url=BASE_URL, api_key=api_key)

In [ ]:
# --- Utilities ---
def is_valid_url(url):
    if not url: return False
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError: return False

def scrape_page_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        for irrelevant in soup.body(["script", "style", "img", "input", "nav", "footer"]):
            irrelevant.decompose()
        return soup.body.get_text(separator="\n", strip=True)[:10000]
    except Exception as e:
        logger.error(f"Error scraping {url}: {e}")
        return ""

def fetch_website_context(url):
    if not is_valid_url(url):
        return "No valid URL provided."
    return scrape_page_text(url)

In [ ]:
# --- Export Functions ---
def export_to_pdf(text):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    
    # Clean markdown-ish characters for simple PDF
    clean_text = text.replace("#", "").replace("*", "").replace("-", "•")
    clean_text = clean_text.encode('latin-1', 'replace').decode('latin-1')
    
    pdf.multi_cell(0, 10, txt=clean_text)
    path = "brochure.pdf"
    pdf.output(path)
    return path

def export_to_docx(text):
    doc = Document()
    for line in text.split('\n'):
        if line.startswith('# '):
            doc.add_heading(line[2:], level=1)
        elif line.startswith('## '):
            doc.add_heading(line[3:], level=2)
        elif line.startswith('### '):
            doc.add_heading(line[4:], level=3)
        else:
            doc.add_paragraph(line)
            
    path = "brochure.docx"
    doc.save(path)
    return path

In [ ]:
# --- Brochure Generation ---
def generate_brochure_content(company_name, url, preferences):
    if not api_key:
        return "❌ API Key not configured."

    context = fetch_website_context(url) if url else "No website context provided."
    
    # Build preference string
    pref_list = []
    if "Bullet Points" in preferences: pref_list.append("Key features and services should be listed as bullet points.")
    if "Paragraphs" in preferences: pref_list.append("Include detailed descriptive paragraphs for the company overview and vision.")
    if "Professional Tone" in preferences: pref_list.append("Use a formal and professional corporate tone.")
    if "Witty/Creative Tone" in preferences: pref_list.append("Use a witty, modern, and creative tone.")
    
    pref_text = "\n".join(pref_list) if pref_list else "Use a standard professional brochure format."

    system_prompt = f"""
    You are a world-class corporate copywriter. Create a professional company brochure.
    
    Guidelines:
    {pref_text}
    
    Crucially:
    - Identify and include relevant technologies, tools, and industry standards the company utilizes or offers.
    - Ensure the brochure has a catchy Headline, 'About Us' section, 'Our Services/Products' section, 'Why Choose Us', and a 'Contact' placeholder.
    """
    
    user_prompt = f"Company Name: {company_name}\nWebsite Context: {context}\n\nPlease generate the brochure content in Markdown."
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        logger.error(f"Generation Error: {e}")
        return f"❌ An error occurred: {e}"

In [ ]:
# --- Gradio Interface ---
def process_all(company_name, url, preferences):
    content = generate_brochure_content(company_name, url, preferences)
    if content.startswith("❌"):
        return content, None, None
    
    pdf_path = export_to_pdf(content)
    docx_path = export_to_docx(content)
    
    return content, pdf_path, docx_path

with gr.Blocks(title="AI Company Brochure App", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏢 AI Company Brochure Generator")
    gr.Markdown("Generate a professional company brochure with customized formatting and export options.")
    
    with gr.Row():
        with gr.Column():
            company_name_input = gr.Textbox(label="Company Name", placeholder="e.g. TechNova AI")
            url_input = gr.Textbox(label="Company Website (Optional)", placeholder="https://example.com")
            
            preferences_input = gr.CheckboxGroup(
                choices=["Bullet Points", "Paragraphs", "Professional Tone", "Witty/Creative Tone"],
                label="Brochure Preferences",
                value=["Bullet Points", "Professional Tone"]
            )
            
            generate_btn = gr.Button("Generate Brochure ✨", variant="primary")
            
        with gr.Column():
            output_markdown = gr.Markdown(label="Generated Preview")
            
            with gr.Row():
                pdf_file = gr.File(label="Download PDF")
                docx_file = gr.File(label="Download DOCX")
    
    generate_btn.click(
        fn=process_all, 
        inputs=[company_name_input, url_input, preferences_input], 
        outputs=[output_markdown, pdf_file, docx_file]
    )

demo.launch(share=True,inbrowser=True,inline=False)